# From Tokens to Circuits — Parte 2: interpretabilidad mecanicista en la práctica

Ahora abrimos un modelo **de verdad**: GPT-2 small (124M parámetros, 12 capas × 12 cabezas). Usamos [TransformerLens](https://github.com/TransformerLensOrg/TransformerLens), una librería que expone cada activación interna con un nombre (*hook point*) para poder **leerla** y **modificarla**.

Las cuatro herramientas básicas que vamos a usar:

| Herramienta | Pregunta que responde |
|---|---|
| **Caché de activaciones** | ¿qué valores toma cada componente? |
| **Logit lens** | ¿qué "cree" el modelo en cada capa intermedia? |
| **Ablación** | si apago este componente, ¿qué se rompe? |
| **Activation patching** | ¿qué componente lleva la información que cambia la respuesta? |

Las celdas `anim(...)` reproducen las animaciones de las diapositivas.

Y dos fenómenos donde aplicarlas: **induction heads** y la tarea **IOI** (*indirect object identification*). Cerramos con **steering**: modificar el comportamiento sumando un vector.

> 💡 Funciona en CPU, pero en Colab con GPU va más fluido.

In [ ]:
!pip -q install "transformer_lens<4"

In [ ]:
import torch
import matplotlib.pyplot as plt
from transformer_lens import HookedTransformer, patching
import transformer_lens.utils as utils

torch.set_grad_enabled(False)   # solo vamos a mirar, no a entrenar
device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2-small", device=device)
cfg = model.cfg
print(f"{cfg.n_layers} capas, {cfg.n_heads} cabezas, d_model={cfg.d_model}, vocab={cfg.d_vocab}")

In [ ]:
#@title 🎨 Estilo del taller (ejecuta esta celda; puedes ignorar su código)
import base64, html, io, logging, re, urllib.request, uuid
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import HTML, display

# Paleta de las diapositivas
INK, MUTED, PAPER, PAPER2, LINE = "#13211A", "#45554C", "#F2F5EF", "#E1EADD", "#DAE4D5"
GREEN, GREEN_D, GREEN_L, BLUE, DARK = "#2FA36B", "#1B7347", "#6BD49C", "#3B82C4", "#0F1B16"

def _load_font(family, weights="400;600;700"):
    try:
        url = "https://fonts.googleapis.com/css2?family=" + family.replace(" ", "+") + ":wght@" + weights
        css = urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "Python"}), timeout=10).read().decode()
        for i, u in enumerate(re.findall(r"url\((https://[^)]+\.ttf)\)", css)):
            path = f"/tmp/{family.replace(' ', '_')}_{i}.ttf"
            open(path, "wb").write(urllib.request.urlopen(u, timeout=20).read())
            font_manager.fontManager.addfont(path)
        return True
    except Exception as e:
        print(f"(No se pudo descargar {family}; se usará la fuente por defecto: {e})")
        return False

logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
_sans = _load_font("Google Sans")
_mono = _load_font("Google Sans Code", "400;600")
mpl.rcParams.update({
    "font.family": ["Google Sans", "DejaVu Sans"] if _sans else ["DejaVu Sans"],
    "font.monospace": ["Google Sans Code", "DejaVu Sans Mono"],
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": LINE, "axes.labelcolor": MUTED, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": False, "grid.color": LINE, "grid.alpha": 1.0,
    "axes.prop_cycle": mpl.cycler(color=[GREEN, BLUE, INK, "#7D8C82", GREEN_L, "#8CBBE3"]),
    "lines.linewidth": 2.4, "lines.solid_capstyle": "round", "legend.frameon": False,
    "figure.dpi": 110, "image.cmap": "taller",
})
for _name, _cols in {
    "taller": [PAPER, "#B5E0C6", GREEN, GREEN_D, DARK],      # secuencial: crema → verde → casi negro
    "taller_div": [BLUE, "#B9D5EF", PAPER, "#9ED6B5", GREEN_D], # divergente: azul ← crema → verde
}.items():
    try: mpl.colormaps.register(LinearSegmentedColormap.from_list(_name, _cols), name=_name)
    except ValueError: pass

# Animaciones de las diapositivas, con el mismo código que en la presentación
def anim(name, max_width=980):
    """Muestra una animación de las diapositivas del taller."""
    w, h, body = ANIMS[name]
    doc = ('<!doctype html><html><head><meta charset="utf-8">'
           '<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Google+Sans:wght@400..700'
           '&family=Google+Sans+Code:wght@400..600&display=swap"></head><body style="margin:0">' + body + '</body></html>')
    uid = "a" + uuid.uuid4().hex[:8]
    s0 = min(1, max_width / w)
    display(HTML(
        f'<div id="{uid}" style="width:100%;max-width:{max_width}px;height:{h*s0:.0f}px;position:relative;overflow:hidden;border-radius:14px">'
        f'<iframe srcdoc="{html.escape(doc, quote=True)}" scrolling="no" '
        f'style="position:absolute;left:0;top:0;width:{w}px;height:{h}px;border:0;transform:scale({s0});transform-origin:0 0"></iframe></div>'
        f'<script>(function(){{var d=document.getElementById("{uid}"),f=d.firstChild;'
        f'function fit(){{var s=Math.min(1,d.clientWidth/{w});f.style.transform="scale("+s+")";d.style.height=({h}*s)+"px"}}'
        f'fit();if(window.ResizeObserver)new ResizeObserver(fit).observe(d);}})();</script>'))

ANIMS = {}
ANIMS['residual'] = (1664, 560, '<style>\nhtml,body{margin:0;height:100%;background:#0F1B16;overflow:hidden;font-family:\'Google Sans\',system-ui,sans-serif;color:#EAF2EA}\n.c{position:absolute;top:20px;width:96px;height:64px;border-radius:12px;display:flex;align-items:center;justify-content:center;font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:26px;background:#1A2C23;color:#A2B8AB;transition:all .3s}\n.c.on{color:#0F1B16}\n.ar{position:absolute;top:92px;width:4px;height:0;transition:height .35s}\n#bus{position:absolute;left:0;top:200px;width:1664px;height:24px;border-radius:12px;background:#1A2C23}\n#stack{position:absolute;left:1440px;top:250px;width:200px;display:flex;flex-direction:column-reverse;gap:3px}\n.seg{height:18px;border-radius:3px}\n#eq{position:absolute;left:0;top:300px;width:1300px;font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:34px;color:#EAF2EA;line-height:1.5}\n#eq span{color:#A2B8AB}\n#chk{position:absolute;left:0;top:440px;width:1300px;font-size:28px;color:#A2B8AB}\n#chk b{color:#6BD49C}\n#tok{position:absolute;top:186px;width:52px;height:52px;border-radius:26px;background:#EAF2EA;transition:left .35s linear}\n</style>\n<div id="bus"></div><div id="tok"></div><div id="stack"></div>\n<div id="eq">x_final = embedding <span>+ Σ salidas de atención</span> + Σ salidas de MLP</div>\n<div id="chk">Comprobado en el notebook 2: reconstruir el residual final sumando todas las piezas da un <b>error de 0,0</b>.</div>\n<script>\nconst N=["emb","A0","M0","A1","M1","A2","M2","···","A10","M10","A11","M11"],col=n=>n=="emb"?"#EAF2EA":n[0]=="A"?"#6FAEE8":n[0]=="M"?"#6BD49C":"#56695E";\nconst cs=N.map((n,i)=>{const d=document.createElement("div");d.className="c";d.style.left=i*118+"px";d.textContent=n;document.body.appendChild(d);\n const a=document.createElement("div");a.className="ar";a.style.left=i*118+46+"px";a.style.background=col(n);document.body.appendChild(a);return [d,a]});\nconst un=document.createElement("div");un.className="c";un.style.left="1440px";un.style.width="200px";un.textContent="→ logits";document.body.appendChild(un);\nconst tok=document.getElementById("tok"),st=document.getElementById("stack");\nfunction run(){st.innerHTML="";cs.forEach(([d,a])=>{d.className="c";d.style.background="#1A2C23";a.style.height="0"});un.style.background="#1A2C23";un.style.color="#A2B8AB";tok.style.left="-60px";\n N.forEach((n,i)=>setTimeout(()=>{tok.style.left=i*118+22+"px";const [d,a]=cs[i];d.classList.add("on");d.style.background=col(n);a.style.height="96px";\n  if(n!="···"){const sg=document.createElement("div");sg.className="seg";sg.style.background=col(n);st.appendChild(sg)}},300+i*480));\n setTimeout(()=>{tok.style.left="1520px";un.style.background="#EAF2EA";un.style.color="#0F1B16"},300+N.length*480)}\nrun();setInterval(run,8600);\n</script>')
ANIMS['logitlens'] = (1664, 620, '<style>\nhtml,body{margin:0;height:100%;background:#F2F5EF;overflow:hidden;font-family:\'Google Sans\',system-ui,sans-serif;color:#13211A}\nsvg{position:absolute;left:0;top:0}\n#pr{position:absolute;left:0;top:0;font-size:30px;color:#45554C}\n#pr b{font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;color:#13211A}\n#panel{position:absolute;left:1120px;top:80px;width:540px;background:#0F1B16;color:#EAF2EA;border-radius:20px;padding:36px;box-sizing:border-box;font-size:26px;color:#A2B8AB}\n#panel #tp,#panel #ly{color:#EAF2EA}\n#panel #ly{color:#6BD49C}\n#ly{font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:28px;color:#6BD49C}\n#tp{font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:52px;font-weight:600;margin:14px 0;white-space:pre}\n#pb{font-size:28px;color:#A2B8AB}\n.lg{position:absolute;left:1120px;font-size:28px}\n</style>\n<div id="pr">Prompt: <b>“Michael Jordan plays the sport of”</b></div>\n<svg id="s" width="1080" height="620"></svg>\n<div id="panel"><div id="ly"></div><div>top-1 si el modelo acabara aquí:</div><div id="tp"></div><div id="pb"></div></div>\n<div class="lg" style="top:440px;color:#22865A">● basketball</div><div class="lg" style="top:484px;color:#2B6699">● football</div><div class="lg" style="top:528px;color:#56665C">● golf</div>\n<script>\nconst B=[0,0,0,0,0,.001,.008,.02,.135,.425,.949,.898,.5],F=[0,0,0,0,0,.003,.054,.144,.14,.093,.022,.041,.045],G=[0,0,0,0,0,0,0,.001,.006,.041,.002,.004,.043];\nconst TOP=[" destro"," the"," the"," the"," the"," the"," course"," the"," baseball"," basketball"," basketball"," basketball"," basketball"];\nconst NS="http://www.w3.org/2000/svg",s=document.getElementById("s"),X=i=>90+i*76,Y=v=>540-v*440;\nfunction el(n,a){const e=document.createElementNS(NS,n);for(const k in a)e.setAttribute(k,a[k]);s.appendChild(e);return e}\n[0,.25,.5,.75,1].forEach(v=>{el("line",{x1:90,y1:Y(v),x2:1002,y2:Y(v),stroke:"#DAE4D5"});const t=el("text",{x:70,y:Y(v)+8,"text-anchor":"end","font-size":24,fill:"#45554C"});t.textContent=Math.round(v*100)+"%"});\nfor(let i=0;i<13;i++){const t=el("text",{x:X(i),y:584,"text-anchor":"middle","font-size":24,fill:"#45554C"});t.textContent=i==12?"fin":i}\nconst tx=el("text",{x:546,y:616,"text-anchor":"middle","font-size":24,fill:"#45554C"});tx.textContent="capa";\nconst cur=el("rect",{x:X(0)-30,y:90,width:60,height:460,rx:10,fill:"#E1EADD"});s.insertBefore(cur,s.children[0]);\nconst mk=(d,c)=>{const p=el("path",{d:"M"+d.map((v,i)=>X(i)+" "+Y(v)).join(" L"),fill:"none",stroke:c,"stroke-width":6,"stroke-linejoin":"round","stroke-linecap":"round"});return p};\nconst P=[mk(G,"#7D8C82"),mk(F,"#2B6699"),mk(B,"#22865A")];\nconst dot=el("circle",{cx:X(0),cy:Y(0),r:12,fill:"#22865A"});\nlet i=0;\nfunction step(){if(i==0)P.forEach(p=>{const L=p.getTotalLength();p.style.transition="none";p.style.strokeDasharray=L;p.style.strokeDashoffset=L});\n const f=i/12;P.forEach(p=>{const L=parseFloat(p.style.strokeDasharray);p.style.transition="stroke-dashoffset .7s linear";p.style.strokeDashoffset=L*(1-f)});\n cur.setAttribute("x",X(i)-30);dot.setAttribute("cx",X(i));dot.setAttribute("cy",Y(B[i]));\n document.getElementById("ly").textContent=i==12?"salida final":"tras la capa "+i;document.getElementById("tp").textContent=TOP[i];\n document.getElementById("pb").textContent="P( basketball) = "+(B[i]*100).toFixed(1)+"%";\n i=i>=12?0:i+1;}\nstep();setInterval(()=>{step();if(i==0)setTimeout(()=>{},0)},900);\n</script>')
ANIMS['inductionidea'] = (1664, 630, '<style>\nhtml,body{margin:0;height:100%;background:#0F1B16;overflow:hidden;font-family:\'Google Sans\',system-ui,sans-serif;color:#EAF2EA}\nsvg{position:absolute;left:0;top:0}\n.t{position:absolute;top:300px;height:68px;border-radius:12px;background:#1A2C23;display:flex;align-items:center;justify-content:center;font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:30px;white-space:pre;transition:background .4s}\n.lbl{position:absolute;font-size:26px;font-weight:600;border-radius:12px;padding:10px 18px;transition:opacity .5s;white-space:nowrap}\n#pred{position:absolute;top:300px;left:1350px;width:300px}\n#pred .k{font-size:24px;color:#A2B8AB;position:absolute;top:-36px;left:0}\n#out{height:68px;border-radius:12px;border:3px dashed #6BD49C;display:flex;align-items:center;justify-content:center;font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:32px;font-weight:600;color:#0F1B16;background:#6BD49C;white-space:pre;transition:all .5s}\n#out.empty{background:transparent;color:transparent}\n#steps{position:absolute;left:0;top:420px;width:1664px;display:flex;gap:20px}\n.s{flex:1;font-size:28px;line-height:1.35;padding:22px 26px;border-radius:16px;background:#1A2C23;color:#A2B8AB;border:3px solid #1A2C23;transition:all .4s}\n.s b{color:#EAF2EA}\n.s.on{border-color:#6BD49C;color:#EAF2EA}\n.num{display:inline-block;width:40px;height:40px;border-radius:20px;text-align:center;line-height:40px;font-weight:700;margin-right:10px;color:#0F1B16}\n</style>\n<svg id="s" width="1664" height="420"></svg>\n<div id="pred"><div class="k">predicción</div><div id="out"> Potter</div></div>\n<div id="steps">\n<div class="s" id="s0"><span class="num" style="background:#6FAEE8">1</span>Una cabeza de <b>token anterior</b> escribe en « Potter»: «vengo detrás de Harry».</div>\n<div class="s" id="s1"><span class="num" style="background:#6BD49C">2</span>La <b>induction head</b>, desde el segundo «Harry», busca al que venía detrás de Harry.</div>\n<div class="s" id="s2"><span class="num" style="background:#EAF2EA">3</span>Lo encuentra en « Potter» y <b>copia</b> ese token como predicción.</div></div>\n<script>\nconst T=["Harry"," Potter"," went"," to"," the"," station","."," Later",","," Harry"],W=[130,140,120,96,110,150,60,130,60,130];\nconst xs=[];let x=0;T.forEach((t,i)=>{const d=document.createElement("div");d.className="t";d.id="t"+i;d.style.left=x+"px";d.style.width=W[i]+"px";d.textContent=t;document.body.appendChild(d);xs.push(x+W[i]/2);x+=W[i]+14});\nconst NS="http://www.w3.org/2000/svg",s=document.getElementById("s");\nconst arc=(a,b,c,h,w)=>{const p=document.createElementNS(NS,"path");p.setAttribute("d",`M${xs[a]} 296 C ${xs[a]} ${296-h} ${xs[b]} ${296-h} ${xs[b]} 296`);p.setAttribute("fill","none");p.setAttribute("stroke",c);p.setAttribute("stroke-width",w);p.setAttribute("stroke-linecap","round");s.appendChild(p);const L=p.getTotalLength();p.style.strokeDasharray=L;p.dataset.L=L;return p};\nconst a1=arc(1,0,"#6FAEE8",110,8),a2=arc(9,1,"#6BD49C",250,8);\nconst cp=document.createElementNS(NS,"path");cp.setAttribute("d",`M${xs[9]+70} 334 L 1340 334`);cp.setAttribute("stroke","#6BD49C");cp.setAttribute("stroke-width",6);cp.setAttribute("stroke-dasharray","10 10");s.appendChild(cp);\nconst lab=(txt,l,t,bg,fg)=>{const d=document.createElement("div");d.className="lbl";d.style.left=l+"px";d.style.top=t+"px";d.style.background=bg;d.style.color=fg;d.textContent=txt;document.body.appendChild(d);return d};\nconst n1=lab("① token anterior (p. ej. L4H11)",0,20,"#1E3A4A","#B9D5EF"),n2=lab("② induction head (p. ej. L5H5)",720,20,"#1F4A33","#C8EAD5");\nconst out=document.getElementById("out");\nconst st=k=>[0,1,2].forEach(i=>document.getElementById("s"+i).classList.toggle("on",i==k));\nconst hl=(i,c)=>document.getElementById("t"+i).style.background=c;\nconst draw=(p,on,anim)=>{p.style.transition=anim?"stroke-dashoffset 1.1s ease-out":"none";p.style.strokeDashoffset=on?0:p.dataset.L};\nfunction full(){draw(a1,1);draw(a2,1);n1.style.opacity=1;n2.style.opacity=1;cp.style.opacity=1;out.classList.remove("empty");hl(0,"#1E3A4A");hl(1,"#2B6699");hl(9,"#22865A");st(-1)}\nfunction run(){draw(a1,0);draw(a2,0);n1.style.opacity=0;n2.style.opacity=0;cp.style.opacity=0;out.classList.add("empty");T.forEach((_,i)=>hl(i,"#1A2C23"));st(-1);\n setTimeout(()=>{st(0);hl(1,"#2B6699");hl(0,"#1E3A4A");draw(a1,1,1);n1.style.opacity=1},700);\n setTimeout(()=>{st(1);hl(9,"#22865A");draw(a2,1,1);n2.style.opacity=1},3400);\n setTimeout(()=>{st(2);cp.style.opacity=1;out.classList.remove("empty")},6100);\n setTimeout(full,9000)}\nfull();setTimeout(()=>{run();setInterval(run,12500)},3500);\n</script>')
ANIMS['patching'] = (1664, 560, '<style>\nhtml,body{margin:0;height:100%;background:#0F1B16;overflow:hidden;font-family:\'Google Sans\',system-ui,sans-serif;color:#EAF2EA}\n.lab{position:absolute;font-size:26px;color:#A2B8AB;width:380px}\n.lab b{color:#EAF2EA;display:block;font-size:28px}\n.tk{position:absolute;font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:24px;color:#A2B8AB;width:120px;text-align:center;white-space:pre}\n.g{position:absolute;display:grid;grid-template-columns:repeat(5,120px);gap:4px}\n.c{height:24px;border-radius:4px;background:#26392F;transition:background .4s}\n.out{position:absolute;width:250px;height:64px;border-radius:12px;display:flex;align-items:center;justify-content:center;font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:30px;font-weight:600;transition:all .5s}\n#fly{position:absolute;width:120px;height:24px;border-radius:4px;background:#6BD49C;opacity:0;transition:top 1s ease-in-out,opacity .3s}\n#meter{position:absolute;left:1200px;top:420px;width:400px}\n#mv{font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:48px;font-weight:600;color:#6BD49C}\n#mt{font-size:26px;color:#A2B8AB}\n#bar{height:16px;border-radius:8px;background:#26392F;margin-top:10px;overflow:hidden}\n#bf{height:16px;width:0;background:#6BD49C;transition:width .6s}\n</style>\n<div class="lab" style="left:1200px;top:40px"><b>Limpio</b>…, John gave a drink to</div>\n<div class="lab" style="left:1200px;top:300px"><b>Corrupto</b>…, Mary gave a drink to</div>\n<div id="fly"></div>\n<div id="meter"><div id="mt">recuperado</div><div id="mv">—</div><div id="bar"><div id="bf"></div></div></div>\n<script>\nconst P=["When…","S2","gave","drink"," to"],Ly=[10,8,6,4,2,0];\nconst mkRun=(top,s2,cl)=>{P.forEach((t,j)=>{const d=document.createElement("div");d.className="tk";d.style.left=250+j*124+"px";d.style.top=top+"px";d.textContent=j==1?s2:t;document.body.appendChild(d)});\n const g=document.createElement("div");g.className="g";g.style.left="250px";g.style.top=top+40+"px";const cs=[];for(let i=0;i<30;i++){const c=document.createElement("div");c.className="c";g.appendChild(c);cs.push(c)}document.body.appendChild(g);\n const o=document.createElement("div");o.className="out";o.style.left="920px";o.style.top=top+70+"px";o.style.background=cl;document.body.appendChild(o);return {cs,o}};\nconst A=mkRun(10,"␣John","#2B6699"),B=mkRun(290,"␣Mary","#22865A");\nA.o.textContent="→ Mary";B.o.textContent="→ John";\nfor(let r=0;r<6;r++){const l=document.createElement("div");l.className="tk";l.style.left="130px";l.style.top=50+r*28+"px";l.style.width="100px";l.textContent="L"+Ly[r];document.body.appendChild(l);\n const l2=l.cloneNode(true);l2.style.top=330+r*28+"px";document.body.appendChild(l2)}\nconst fly=document.getElementById("fly");\nconst cases=[[4,1,1.0,"L2 en S2"],[4,4,0.0,"L2 en «to»"],[1,1,0.63,"L8 en S2"],[0,4,1.0,"L10 en «to»"]];let k=0;\nfunction run(){const [r,c,v,name]=cases[k];[A,B].forEach(R=>R.cs.forEach(x=>x.style.background="#26392F"));\n B.o.textContent="→ John";B.o.style.background="#22865A";document.getElementById("mv").textContent="—";document.getElementById("bf").style.width="0";document.getElementById("mt").textContent="parcheo: "+name;\n const i=r*6+c-r;const idx=r*5+c;A.cs[idx].style.background="#6BD49C";\n fly.style.transition="none";fly.style.left=250+c*124+"px";fly.style.top=50+r*28+"px";fly.style.opacity=1;fly.getBoundingClientRect();fly.style.transition="top 1s ease-in-out,opacity .3s";\n setTimeout(()=>{fly.style.top=330+r*28+"px"},600);\n setTimeout(()=>{fly.style.opacity=0;B.cs[idx].style.background="#6BD49C";document.getElementById("mv").textContent=v.toFixed(2);document.getElementById("bf").style.width=v*100+"%";\n  if(v>0.5){B.o.textContent="→ Mary";B.o.style.background="#2B6699"}},1700);\n k=(k+1)%cases.length}\nrun();setInterval(run,4200);\n</script>')
ANIMS['ioi'] = (1664, 560, '<style>\nhtml,body{margin:0;height:100%;background:#0F1B16;overflow:hidden;font-family:\'Google Sans\',system-ui,sans-serif;color:#EAF2EA}\nsvg{position:absolute;left:0;top:0}\n.b{position:absolute;width:360px;border-radius:16px;padding:20px 24px;box-sizing:border-box;background:#1A2C23;border:3px solid #26392F;transition:all .4s}\n.b.on{border-color:#6BD49C;background:#173A2A}\n.t{font-size:30px;font-weight:600}\n.h{font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:24px;color:#6FAEE8;margin-top:6px}\n.d{font-size:24px;color:#A2B8AB;margin-top:8px;line-height:1.3}\n#ans{position:absolute;left:1440px;top:150px;width:220px;height:90px;border-radius:16px;background:#2B6699;display:flex;align-items:center;justify-content:center;font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:36px;font-weight:600;opacity:.25;transition:opacity .5s}\n.pulse{position:absolute;width:22px;height:22px;border-radius:11px;background:#6BD49C;opacity:0}\n</style>\n<svg id="s" width="1664" height="560"></svg>\n<div class="b" id="b0" style="left:0;top:60px"><div class="t">Detectar duplicado</div><div class="h">0.1 · 3.0 · 5.5 · 6.9</div><div class="d">En el 2º «John»: «este nombre ya salió»</div></div>\n<div class="b" id="b1" style="left:520px;top:60px"><div class="t">S-inhibition</div><div class="h">7.3 · 7.9 · 8.6 · 8.10</div><div class="d">Mueven al final: «no copies John»</div></div>\n<div class="b" id="b2" style="left:1040px;top:60px"><div class="t">Name movers</div><div class="h">9.9 · 9.6 · 10.0</div><div class="d">Atienden al nombre no inhibido y lo copian</div></div>\n<div class="b" id="b3" style="left:1040px;top:330px"><div class="t">Negative name movers</div><div class="h">10.7 · 11.10</div><div class="d">Empujan en contra: autocorrección</div></div>\n<div id="ans">Mary</div>\n<script>\nconst NS="http://www.w3.org/2000/svg",s=document.getElementById("s");\nconst ln=(x1,y1,x2,y2,c,dash)=>{const l=document.createElementNS(NS,"line");[["x1",x1],["y1",y1],["x2",x2],["y2",y2],["stroke",c],["stroke-width",5]].forEach(([k,v])=>l.setAttribute(k,v));if(dash)l.setAttribute("stroke-dasharray","12 10");s.appendChild(l)};\nln(360,140,520,140,"#3A5046");ln(880,140,1040,140,"#3A5046");ln(1400,180,1440,195,"#3A5046");ln(1220,260,1220,330,"#3A5046",1);ln(1400,400,1500,240,"#3A5046",1);\nconst pulse=(x1,y1,x2,y2,t0)=>{const p=document.createElement("div");p.className="pulse";document.body.appendChild(p);\n setTimeout(()=>{p.style.transition="none";p.style.left=x1-11+"px";p.style.top=y1-11+"px";p.style.opacity=1;p.getBoundingClientRect();p.style.transition="left .9s ease-in-out,top .9s ease-in-out,opacity .3s";p.style.left=x2-11+"px";p.style.top=y2-11+"px";setTimeout(()=>p.style.opacity=0,950)},t0)};\nconst on=(i,t)=>setTimeout(()=>{document.getElementById("b"+i).classList.add("on")},t);\nfunction run(){[0,1,2,3].forEach(i=>document.getElementById("b"+i).classList.remove("on"));document.getElementById("ans").style.opacity=.25;\n on(0,300);pulse(360,140,520,140,1200);on(1,2100);pulse(880,140,1040,140,3000);on(2,3900);pulse(1400,180,1450,195,4800);\n setTimeout(()=>document.getElementById("ans").style.opacity=1,5700);on(3,6500);pulse(1220,260,1220,330,6500)}\nrun();setInterval(run,10000);\n</script>')
ANIMS['features'] = (1664, 620, '<style>\nhtml,body{margin:0;height:100%;background:#F2F5EF;overflow:hidden;font-family:\'Google Sans\',system-ui,sans-serif;color:#13211A}\nsvg{position:absolute;left:0;top:0}\n.p{position:absolute;font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:30px;font-weight:600}\n#txt{position:absolute;left:900px;top:30px;width:740px}\n.s{font-size:30px;line-height:1.4;padding:18px 24px;border-radius:14px;margin-bottom:16px;color:#7D8C82;transition:all .4s}\n.s.on{background:#0F1B16;color:#EAF2EA}\n#f{position:absolute;left:900px;top:470px;width:740px;font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:30px;background:#E1EADD;border-radius:14px;padding:20px 24px;box-sizing:border-box}\n</style>\n<svg id="s" width="860" height="620"></svg>\n<div class="p" style="left:120px;top:470px">hombre</div><div class="p" style="left:120px;top:150px">mujer</div>\n<div class="p" style="left:620px;top:470px">rey</div><div class="p" style="left:620px;top:150px">reina</div>\n<div id="txt"><div class="s" id="s0">Mismo desplazamiento hombre → mujer y rey → reina: una dirección de «género»</div>\n<div class="s" id="s1">Otra dirección, ortogonal: «realeza»</div>\n<div class="s" id="s2">Una activación ≈ suma de features × intensidad</div></div>\n<div id="f">a ≈ 0,9·realeza + 0,8·femenino + …</div>\n<script>\nconst NS="http://www.w3.org/2000/svg",s=document.getElementById("s");\nfunction el(n,a){const e=document.createElementNS(NS,n);for(const k in a)e.setAttribute(k,a[k]);s.appendChild(e);return e}\nconst d=el("defs",{});["o","b","k"].forEach((id,i)=>{const m=el("marker",{id:id,viewBox:"0 0 10 10",refX:8,refY:5,markerWidth:5,markerHeight:5,orient:"auto"});d.appendChild(m);const p=document.createElementNS(NS,"path");p.setAttribute("d","M0 0 L10 5 L0 10 z");p.setAttribute("fill",["#2FA36B","#3B82C4","#13211A"][i]);m.appendChild(p)});\nconst P={h:[160,440],m:[160,200],r:[660,440],q:[660,200]};\nfor(let i=0;i<9;i++){el("line",{x1:40,y1:40+i*65,x2:820,y2:40+i*65,stroke:"#DAE4D5"});}\nObject.values(P).forEach(p=>el("circle",{cx:p[0],cy:p[1],r:16,fill:"#13211A"}));\nconst ar=(a,b,c,m)=>{const l=el("line",{x1:a[0],y1:a[1],x2:b[0],y2:b[1],stroke:c,"stroke-width":8,"marker-end":`url(#${m})`,"stroke-linecap":"round"});const L=Math.hypot(b[0]-a[0],b[1]-a[1]);l.style.strokeDasharray=L;l.style.strokeDashoffset=L;l.style.transition="stroke-dashoffset 1s ease-out";return l};\nconst sh=(a,b)=>[a[0]+(b[0]-a[0])*0.1,a[1]+(b[1]-a[1])*0.1],en=(a,b)=>[a[0]+(b[0]-a[0])*0.88,a[1]+(b[1]-a[1])*0.88];\nconst g1=ar(sh(P.h,P.m),en(P.h,P.m),"#2FA36B","o"),g2=ar(sh(P.r,P.q),en(P.r,P.q),"#2FA36B","o");\nconst r1=ar(sh(P.h,P.r),en(P.h,P.r),"#3B82C4","b"),r2=ar(sh(P.m,P.q),en(P.m,P.q),"#3B82C4","b");\nconst v=ar([160,560],[660,200],"#13211A","k");v.setAttribute("stroke-dasharray","1");\nconst st=k=>[0,1,2].forEach(i=>document.getElementById("s"+i).className="s"+(i==k?" on":""));\nconst show=(l,on)=>{l.style.transition="stroke-dashoffset 1s ease-out, opacity .25s";l.style.opacity=on?1:0;l.style.strokeDashoffset=on?0:l.style.strokeDasharray};\nfunction run(){[g1,g2,r1,r2].forEach(l=>show(l,0));v.style.opacity=0;st(-1);\n setTimeout(()=>{st(0);show(g1,1);setTimeout(()=>show(g2,1),600)},400);\n setTimeout(()=>{st(1);show(r1,1);setTimeout(()=>show(r2,1),600)},3200);\n setTimeout(()=>{st(2)},6000)}\nv.remove();run();setInterval(run,9500);\n</script>')
ANIMS['steering'] = (1664, 620, '<style>\nhtml,body{margin:0;height:100%;background:#F2F5EF;overflow:hidden;font-family:\'Google Sans\',system-ui,sans-serif;color:#13211A}\nsvg{position:absolute;left:0;top:40px}\n.f{position:absolute;left:0;top:0;width:560px;font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:28px;line-height:1.4}\n.f span{color:#22865A;font-weight:600}\n#alpha{position:absolute;left:0;top:520px;width:560px;font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:44px;font-weight:600}\n.col{position:absolute;top:0;width:500px}\n.hd{font-size:28px;font-weight:600;margin-bottom:14px}\n.o{font-size:26px;line-height:1.4;background:#E1EADD;border-radius:14px;padding:18px 22px;margin-bottom:16px;min-height:110px;white-space:pre-wrap}\n.o i{font-style:normal;color:#56665C}\n</style>\n<div class="f">v = h(<span>“ Love”</span>) − h(“ Hate”)<br>capa 6 · h ← h + α·v</div>\n<svg id="s" width="560" height="480"></svg>\n<div id="alpha"></div>\n<div class="col" style="left:640px"><div class="hd">α = 0 (sin tocar)</div><div class="o" id="a0"></div><div class="o" id="a1"></div></div>\n<div class="col" style="left:1160px"><div class="hd" style="color:#22865A">α = 1,5</div><div class="o" id="b0"></div><div class="o" id="b1"></div></div>\n<script>\nconst NS="http://www.w3.org/2000/svg",s=document.getElementById("s");\nfunction el(n,a){const e=document.createElementNS(NS,n);for(const k in a)e.setAttribute(k,a[k]);s.appendChild(e);return e}\nconst d=el("defs",{});["k","o","b"].forEach((id,i)=>{const m=document.createElementNS(NS,"marker");[["id",id],["viewBox","0 0 10 10"],["refX",8],["refY",5],["markerWidth",4],["markerHeight",4],["orient","auto"]].forEach(([a,b])=>m.setAttribute(a,b));const p=document.createElementNS(NS,"path");p.setAttribute("d","M0 0 L10 5 L0 10 z");p.setAttribute("fill",["#13211A","#22865A","#2B6699"][i]);m.appendChild(p);d.appendChild(m)});\nconst O=[80,400],H=[300,330],V=[140,-170];\nel("line",{x1:O[0],y1:O[1],x2:H[0],y2:H[1],stroke:"#13211A","stroke-width":8,"marker-end":"url(#k)"});\nconst va=el("line",{x1:H[0],y1:H[1],x2:H[0],y2:H[1],stroke:"#22865A","stroke-width":8,"marker-end":"url(#o)"});\nconst nv=el("line",{x1:O[0],y1:O[1],x2:H[0],y2:H[1],stroke:"#2B6699","stroke-width":5,"stroke-dasharray":"12 10","marker-end":"url(#b)"});\nconst t1=el("text",{x:180,y:400,"font-size":28,fill:"#13211A"});t1.textContent="h";\nconst P="I went up to my friend and said";\nconst A=[", “Hey, you guys are going to the movie theater the next day.”",", “I’d like to talk to you about this.” She really looked at me like I was angry"],B=[" this is a big shock! I was excited!!"," I had a hot tea and happy! Really excited! I am really happy"];\nconst typ=(id,t,d0)=>{const e=document.getElementById(id);e.innerHTML="<i></i><span></span>";e.children[0].textContent=P;let i=0;setTimeout(()=>{const iv=setInterval(()=>{e.children[1].textContent=t.slice(0,++i);if(i>=t.length)clearInterval(iv)},30)},d0)};\nfunction setA(a){const x=H[0]+V[0]*a/1.5,y=H[1]+V[1]*a/1.5;va.setAttribute("x2",x);va.setAttribute("y2",y);nv.setAttribute("x2",x);nv.setAttribute("y2",y);document.getElementById("alpha").textContent="α = "+a.toFixed(1)}\nfunction run(){setA(0);typ("a0",A[0],300);typ("a1",A[1],300);["b0","b1"].forEach(i=>document.getElementById(i).innerHTML="");\n let a=0;const iv=setInterval(()=>{a=Math.min(1.5,a+0.05);setA(a);if(a>=1.5){clearInterval(iv);typ("b0",B[0],200);typ("b1",B[1],200)}},70)}\nrun();setInterval(run,11000);\n</script>')
ANIMS['superposition'] = (1664, 620, '<style>\nhtml,body{margin:0;height:100%;background:#F2F5EF;overflow:hidden;font-family:\'Google Sans\',system-ui,sans-serif;color:#13211A}\nsvg{position:absolute;left:60px;top:40px}\n#lab{position:absolute;left:60px;top:0;width:560px;text-align:center;font-size:30px;font-weight:600}\n#sub{position:absolute;left:60px;top:590px;width:560px;text-align:center;font-size:26px;color:#45554C}\n#r{position:absolute;left:760px;top:20px;width:880px}\n.k{font-size:30px;line-height:1.45;color:#45554C;margin-bottom:26px}\n.k b{color:#13211A}\n.neu{background:#0F1B16;color:#EAF2EA;border-radius:18px;padding:28px 32px}\n.neu .h{font-family:\'Google Sans Code\',ui-monospace,Menlo,monospace;font-size:28px;color:#6BD49C;margin-bottom:14px}\n.tag{display:inline-block;font-size:28px;background:#1A2C23;border-radius:10px;padding:8px 14px;margin:6px 8px 0 0;opacity:0;transition:opacity .5s}\n</style>\n<div id="lab"></div>\n<svg id="s" width="560" height="540"></svg><div id="sub"></div>\n<div id="r"><div class="k">Cuando las features son <b>escasas</b> (casi nunca se activan a la vez), el modelo puede meter 5 en 2 dimensiones con un poco de interferencia.</div>\n<div class="k">Consecuencia: una neurona no corresponde a un concepto. Es <b>polisemántica</b>.</div>\n<div class="neu"><div class="h">neurona hipotética #42 se activa con:</div><span class="tag">código Python</span><span class="tag">recetas de cocina</span><span class="tag">texto en coreano</span></div></div>\n<script>\nconst NS="http://www.w3.org/2000/svg",s=document.getElementById("s");\nfunction el(n,a){const e=document.createElementNS(NS,n);for(const k in a)e.setAttribute(k,a[k]);s.appendChild(e);return e}\nconst cx=280,cy=270,R=210,C=["#2FA36B","#3B82C4","#13211A","#1D6E4A","#7DA9D6"];\nel("circle",{cx,cy,r:R,fill:"none",stroke:"#DAE4D5","stroke-width":2});el("line",{x1:cx-R,y1:cy,x2:cx+R,y2:cy,stroke:"#DAE4D5"});el("line",{x1:cx,y1:cy-R,x2:cx,y2:cy+R,stroke:"#DAE4D5"});\nconst L=C.map(c=>el("line",{x1:cx,y1:cy,x2:cx,y2:cy,stroke:c,"stroke-width":10,"stroke-linecap":"round"}));\nconst D=C.map(c=>el("circle",{cx,cy,r:0,fill:c}));\nconst dense=[[1,0],[0,-1],[0,0],[0,0],[0,0]],pent=[0,1,2,3,4].map(i=>{const a=-Math.PI/2+i*2*Math.PI/5;return [Math.cos(a),Math.sin(a)]});\nlet cur=dense.map(v=>v.slice());\nfunction go(T,ms){const st=cur.map(v=>v.slice()),t0=performance.now();\n function f(t){const u=Math.min(1,(t-t0)/ms),e=u<.5?2*u*u:1-Math.pow(-2*u+2,2)/2;\n  cur=st.map((v,i)=>[v[0]+(T[i][0]-v[0])*e,v[1]+(T[i][1]-v[1])*e]);\n  cur.forEach((v,i)=>{const x=cx+v[0]*R*0.92,y=cy+v[1]*R*0.92,on=Math.hypot(v[0],v[1])>0.05;L[i].setAttribute("x2",x);L[i].setAttribute("y2",y);D[i].setAttribute("cx",x);D[i].setAttribute("cy",y);D[i].setAttribute("r",on?16:0)});\n  if(u<1)requestAnimationFrame(f)}requestAnimationFrame(f)}\nconst lab=document.getElementById("lab"),sub=document.getElementById("sub"),tags=[...document.querySelectorAll(".tag")];\nfunction run(){lab.textContent="Features densas: 2 de 5 caben";sub.textContent="las otras 3 se pierden";go(dense,900);tags.forEach(t=>t.style.opacity=0);\n setTimeout(()=>{lab.textContent="Features escasas: caben las 5";sub.textContent="casi ortogonales, con interferencia";go(pent,1800)},3000);\n tags.forEach((t,i)=>setTimeout(()=>t.style.opacity=1,5200+i*500))}\nrun();setInterval(run,10000);\n</script>')
print("Estilo del taller listo ·", "Google Sans" if _sans else "fuente por defecto", "·", len(ANIMS), "animaciones")

In [ ]:
def heatmap(z, title="", xlabel="cabeza", ylabel="capa", cmap="taller_div", center=True, xticklabels=None, figsize=(6, 5)):
    z = z.detach().cpu()
    vmax = z.abs().max().item() if center else None
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(z, cmap=cmap, vmin=-vmax if center else 0, vmax=vmax, aspect="auto")
    cb = fig.colorbar(im, ax=ax, shrink=0.8); cb.outline.set_visible(False)
    ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.set_yticks(range(z.shape[0]))
    if xticklabels is not None: ax.set_xticks(range(len(xticklabels)), xticklabels, rotation=90)
    else: ax.set_xticks(range(z.shape[1]))
    ax.tick_params(length=0); [sp.set_visible(False) for sp in ax.spines.values()]
    plt.show()

## 1. El modelo como un sistema que podemos inspeccionar

`run_with_cache` ejecuta el modelo y guarda **todas** las activaciones intermedias.

In [ ]:
prompt = "Michael Jordan plays the sport of"
tokens = model.to_tokens(prompt)              # añade el token <|endoftext|> al inicio (BOS)
print(model.to_str_tokens(prompt))

logits, cache = model.run_with_cache(tokens)
top = logits[0, -1].softmax(-1).topk(5)
for p, i in zip(top.values, top.indices):
    print(f"{model.to_string(i)!r:12} {p:.3f}")

In [ ]:
# Algunos de los ~200 nombres disponibles en la caché
names = list(cache.keys())
print(len(names))
for n in names[:16]: print(f"{n:35} {tuple(cache[n].shape)}")

**Cómo leer los nombres:**
- `hook_resid_pre / mid / post`: el **residual stream** antes de la capa, tras la atención, tras el MLP.
- `attn.hook_pattern`: los pesos de atención `[batch, cabeza, consulta, clave]`.
- `attn.hook_z`: la salida de cada cabeza **antes** de mezclarse con $W_O$.
- `hook_attn_out`, `hook_mlp_out`: lo que cada subcapa **suma** al residual stream.

Recuerda la parte 1: `x = x + attn(...)`, `x = x + mlp(...)`. El residual stream final es literalmente la **suma** del embedding más las salidas de todas las capas.

In [ ]:
anim("residual")

In [ ]:
# Comprobación: resid final = embeddings + suma de todas las salidas de atención y MLP
resid_final = cache["blocks.11.hook_resid_post"]
recon = cache["hook_embed"] + cache["hook_pos_embed"]
for l in range(cfg.n_layers):
    recon = recon + cache[f"blocks.{l}.hook_attn_out"] + cache[f"blocks.{l}.hook_mlp_out"]
print("error máximo:", (resid_final - recon).abs().max().item())

### ¿Qué aprende un embedding? (la pregunta de la parte 1)

En la parte 1, los vecinos del embedding de nuestro mini-GPT salían casi aleatorios. Hagamos lo mismo con la tabla de embeddings de GPT-2 (`model.W_E`, 50 257 × 768):

In [ ]:
import torch.nn.functional as F
E = F.normalize(model.W_E, dim=1)
def neighbours(word, k=6):
    i = model.to_single_token(word)
    sims = E @ E[i]
    best = sims.topk(k + 1).indices[1:]
    return [(model.to_string(j), round(sims[j].item(), 2)) for j in best]

for w in [" king", " father", " Rome", " basketball"]:   # 🧪 prueba otras (un solo token, con espacio delante)
    print(f"{w!r:13} →", neighbours(w))

Ahora sí: *king* queda cerca de *queen*, *prince* o *monarch*, y *Rome* cerca de *Italy*, *Athens* o *Vatican*. Esa geometría es la materia prima con la que trabaja el resto del modelo.

## 2. Logit lens: ¿cuándo "sabe" el modelo la respuesta?

Truco: cogemos el residual stream **a mitad del modelo**, le aplicamos la LayerNorm final y el *unembedding*, como si el modelo terminara ahí. Así vemos cómo evoluciona la predicción capa a capa.

In [ ]:
anim("logitlens")

In [ ]:
def logit_lens(prompt, answers):
    _, cache = model.run_with_cache(prompt)
    # residual en la última posición tras cada capa: [n_layers+1, 1, d_model]
    resid, labels = cache.accumulated_resid(layer=-1, incl_mid=False, pos_slice=-1, apply_ln=True, return_labels=True)
    probs = model.unembed(resid).softmax(-1)[:, 0]          # [capas, vocab]
    plt.figure(figsize=(7, 3.5))
    for a in answers:
        plt.plot(probs[:, model.to_single_token(a)].cpu(), marker="o", label=repr(a))
    plt.xticks(range(len(labels)), labels, rotation=90); plt.ylabel("probabilidad")
    plt.title(prompt); plt.legend(); plt.grid(axis="y"); plt.show()
    for l in [0, 4, 8, len(labels) - 1]:
        print(f"{labels[l]:>12}: top-5 =", [model.to_string(i) for i in probs[l].topk(5).indices])

logit_lens("Michael Jordan plays the sport of", [" basketball", " football", " golf"])

In [ ]:
# 🧪 Prueba tus propios hechos
logit_lens("The Statue of Liberty is located in the city of New", [" York", " Orleans", " Jersey"])

Compara los dos casos: *New → York* es casi una asociación entre tokens vecinos y aparece ya hacia la capa 4; recuperar un **hecho** (el deporte de Michael Jordan) necesita llegar a las capas medias-tardías. Las primeras capas construyen contexto y las posteriores lo convierten en una predicción concreta.

⚠️ El logit lens es una aproximación: las capas intermedias no "hablan" exactamente el idioma del unembedding, por eso las primeras capas dan tokens sin sentido.

## 3. Patrones de atención

En la parte 1 vimos las cabezas de nuestro mini-GPT. Hagamos lo mismo con GPT-2, y busquemos cabezas con funciones reconocibles.

In [ ]:
def plot_heads(prompt, layer, heads):
    _, cache = model.run_with_cache(prompt)
    labels = model.to_str_tokens(prompt)
    pat = cache["pattern", layer][0]                         # [heads, q, k]
    fig, axes = plt.subplots(1, len(heads), figsize=(4 * len(heads), 4))
    for ax, h in zip(axes, heads):
        ax.imshow(pat[h].cpu(), cmap="taller", vmin=0, vmax=1); ax.tick_params(length=0)
        ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=90, fontsize=7)
        ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=7)
        ax.set_title(f"L{layer}H{h}")
    plt.tight_layout(); plt.show()

plot_heads("When Mary and John went to the store, John gave a drink to", layer=4, heads=[4, 11])  # 🧪

- **L4H11** es una cabeza de **token anterior** casi perfecta: cada token mira al de justo antes.
- **L4H4** mira sobre todo al primer token (`<|endoftext|>`): es un **sumidero de atención**, el sitio donde aparcar atención cuando la cabeza "no tiene nada que hacer".

🧪 Explora otras capas y cabezas. ¿Encuentras alguna que mire siempre a la puntuación, o al mismo token repetido?

In [ ]:
# Puntuación de "token anterior" para las 144 cabezas
text = "The quick brown fox jumps over the lazy dog. Mechanistic interpretability tries to reverse engineer neural networks into human-understandable algorithms."
_, cache = model.run_with_cache(text)
prev = torch.stack([cache["pattern", l][0].diagonal(offset=-1, dim1=-2, dim2=-1).mean(-1) for l in range(cfg.n_layers)])
heatmap(prev, "Atención media al token anterior", cmap="taller", center=False)

## 4. Induction heads

Un **circuito** famoso (Olsson et al., 2022). Si el texto contiene `... A B ... A`, el modelo predice `B`: copia lo que vino después la vez anterior. Está implementado por dos cabezas que colaboran (míralo en la animación y luego lee los pasos):

1. Una **cabeza de token anterior** (capa temprana) escribe en la posición de `B` la información "vengo detrás de `A`".
2. Una **induction head** (capa posterior) desde el segundo `A` busca la posición cuyo token anterior era `A` → atiende a `B` → lo copia.

Experimento limpio: una secuencia de **tokens aleatorios** repetida dos veces. La primera mitad es impredecible; la segunda, trivial si sabes copiar.

In [ ]:
anim("inductionidea")

In [ ]:
torch.manual_seed(0)
seq_len, batch = 50, 8
rand = torch.randint(1000, 10000, (batch, seq_len))
bos = torch.full((batch, 1), model.tokenizer.bos_token_id)
rep_tokens = torch.cat([bos, rand, rand], dim=1).to(device)       # [batch, 1 + 2*seq_len]

rep_logits, rep_cache = model.run_with_cache(rep_tokens)
loss_per_tok = model.loss_fn(rep_logits, rep_tokens, per_token=True).mean(0).cpu()

plt.figure(figsize=(7, 3))
plt.axvspan(seq_len, len(loss_per_tok), color=PAPER2, lw=0)
plt.plot(loss_per_tok, color=BLUE); plt.axvline(seq_len, color=INK, ls="--", lw=1.2)
plt.xlabel("posición"); plt.ylabel("pérdida"); plt.title("Pérdida por token en una secuencia repetida"); plt.grid(axis="y"); plt.show()
print(f"1ª mitad: {loss_per_tok[:seq_len].mean():.2f}  |  2ª mitad: {loss_per_tok[seq_len:].mean():.2f}")

Cuando la secuencia empieza a repetirse, la pérdida cae casi a cero. Busquemos las cabezas responsables: una induction head, desde la posición $i$ de la segunda mitad, mira a la posición $i - \text{seq\_len} + 1$ (el token **siguiente** a la copia anterior).

In [ ]:
ind_score = torch.stack([
    rep_cache["pattern", l].diagonal(offset=-(seq_len - 1), dim1=-2, dim2=-1)[..., 1:].mean(dim=(0, -1))
    for l in range(cfg.n_layers)
])
heatmap(ind_score, "Puntuación de inducción", cmap="taller", center=False)
top = ind_score.flatten().topk(6)
induction_heads = [(i // cfg.n_heads, i % cfg.n_heads) for i in top.indices.tolist()]
print("Top induction heads (capa, cabeza):", induction_heads)

In [ ]:
# Así se ve una induction head: la diagonal desplazada en la segunda mitad
l, h = induction_heads[0]
plt.figure(figsize=(5, 5))
plt.imshow(rep_cache["pattern", l][0, h].cpu(), cmap="taller", vmin=0, vmax=1)
plt.title(f"L{l}H{h} sobre la secuencia repetida"); plt.xlabel("clave"); plt.ylabel("consulta"); plt.show()

## 5. Ablación: ¿son *necesarias* esas cabezas?

Ver un patrón no demuestra que se use. Para una afirmación **causal**, intervenimos: ponemos a cero la salida (`hook_z`) de esas cabezas y medimos la pérdida otra vez.

Los *hooks* de TransformerLens son funciones que reciben la activación y pueden devolverla modificada.

In [ ]:
def ablate_heads_hook(heads_in_layer):
    def hook(z, hook):            # z: [batch, pos, head, d_head]
        z[:, :, heads_in_layer] = 0.0
        return z
    return hook

def loss_with_ablation(heads):
    by_layer = {}
    for l, h in heads: by_layer.setdefault(l, []).append(h)
    hooks = [(utils.get_act_name("z", l), ablate_heads_hook(hs)) for l, hs in by_layer.items()]
    logits = model.run_with_hooks(rep_tokens, fwd_hooks=hooks)
    return model.loss_fn(logits, rep_tokens, per_token=True).mean(0).cpu()

torch.manual_seed(1)
random_heads = [(int(l), int(h)) for l, h in zip(torch.randint(0, 12, (6,)), torch.randint(0, 12, (6,)))]

plt.figure(figsize=(7, 3))
plt.plot(loss_per_tok, color=BLUE, label="original")
plt.plot(loss_with_ablation(random_heads), color="#7D8C82", lw=1.8, label="6 cabezas aleatorias apagadas")
plt.plot(loss_with_ablation(induction_heads), color=GREEN, label="6 induction heads apagadas")
plt.axvline(seq_len, color=INK, ls="--", lw=1.2); plt.legend(); plt.xlabel("posición"); plt.ylabel("pérdida"); plt.grid(axis="y"); plt.show()

Apagar 6 de 144 cabezas elegidas al azar casi no cambia nada; apagar las 6 induction heads **destroza** la capacidad de copiar. (No llega a recuperar del todo la pérdida inicial: hay redundancia y otros mecanismos. Esto es muy típico.)

🧪 ¿Qué pasa si apagas solo la mejor cabeza? ¿Y las cabezas de token anterior de la sección 3?

## 6. Activation patching: localizar la información (tarea IOI)

> *When Mary and John went to the store, John gave a drink to* → **Mary**

Para resolverlo, el modelo debe detectar qué nombre está **repetido** (John) y predecir el **otro**. Medimos el rendimiento con la **diferencia de logits** `logit(Mary) − logit(John)`.

**Activation patching**: ejecutamos el modelo con un prompt *corrupto* (donde la respuesta cambia) y copiamos una activación del prompt *limpio* en un sitio concreto. Si eso basta para recuperar la respuesta correcta, ese sitio lleva la información relevante.

In [ ]:
anim("patching")

In [ ]:
clean_prompt   = "When Mary and John went to the store, John gave a drink to"
corrupt_prompt = "When Mary and John went to the store, Mary gave a drink to"   # cambia el sujeto repetido
clean_tokens, corrupt_tokens = model.to_tokens(clean_prompt), model.to_tokens(corrupt_prompt)
assert clean_tokens.shape == corrupt_tokens.shape

mary, john = model.to_single_token(" Mary"), model.to_single_token(" John")
def logit_diff(logits):
    return logits[0, -1, mary] - logits[0, -1, john]

clean_logits, clean_cache = model.run_with_cache(clean_tokens)
corrupt_logits = model(corrupt_tokens)
CLEAN, CORRUPT = logit_diff(clean_logits).item(), logit_diff(corrupt_logits).item()
print(f"limpio: {CLEAN:.2f}   corrupto: {CORRUPT:.2f}")

def metric(logits):   # 1 = rendimiento limpio, 0 = rendimiento corrupto
    return (logit_diff(logits) - CORRUPT) / (CLEAN - CORRUPT)

### 6.1 Parcheando el residual stream (capa × posición)

In [ ]:
resid_patch = patching.get_act_patch_resid_pre(model, corrupt_tokens, clean_cache, metric)
heatmap(resid_patch, "Parcheo de resid_pre (1 = recupera la respuesta limpia)",
        xlabel="posición", xticklabels=model.to_str_tokens(clean_tokens[0]), figsize=(9, 5))

**Cómo se lee:** en las primeras capas la información está en la posición del **segundo nombre** (` John` / ` Mary`). A partir de la capa ~7-8 "salta" a la **última posición** (` to`). Algo en las capas medias **mueve** información entre posiciones… y lo único que mueve información entre posiciones es la **atención**.

In [ ]:
head_patch = patching.get_act_patch_attn_head_out_all_pos(model, corrupt_tokens, clean_cache, metric)
heatmap(head_patch, "Parcheo de la salida de cada cabeza (todas las posiciones)")
top = head_patch.flatten().abs().topk(8)
print("Cabezas más importantes:", [(i // 12, i % 12, round(head_patch.flatten()[i].item(), 2)) for i in top.indices.tolist()])

Has redescubierto partes del circuito IOI de Wang et al. (2022). Así encajan las piezas:

In [ ]:
anim("ioi")

Las clases de cabezas:

- **Name movers** (≈ capas 9–10, p. ej. 9.9, 9.6, 10.0): atienden al nombre correcto y lo copian a la salida.
- **S-inhibition heads** (≈ 7.3, 7.9, 8.6, 8.10): le dicen a los name movers que **no** miren al nombre repetido.
- **Negative name movers** (p. ej. 10.7, 11.10): valores con signo contrario, empujan *en contra* de la respuesta (¡el modelo se autocorrige!).

🧪 Mira el patrón de atención de un name mover desde la última posición: `plot_heads(clean_prompt, 9, [9, 6])`. ¿A qué nombre atiende?

In [ ]:
plot_heads(clean_prompt, 9, [9, 6])

## 7. Steering: cambiar el comportamiento con un vector

Si los conceptos están codificados como **direcciones** en el residual stream (como *rey − hombre + mujer ≈ reina*), podemos sumar una dirección para empujar al modelo. La receta más simple (*activation addition*, Turner et al., 2023):

1. Activación de `" Love"` menos activación de `" Hate"` en una capa intermedia → vector de dirección.
2. Sumarlo (multiplicado por un coeficiente) al residual stream mientras generamos.

In [ ]:
anim("features")

In [ ]:
anim("steering")

In [ ]:
STEER_LAYER = 6
hook_name = utils.get_act_name("resid_pre", STEER_LAYER)

def direction(pos_word, neg_word):
    a = model.run_with_cache(pos_word, names_filter=hook_name)[1][hook_name][0, -1]
    b = model.run_with_cache(neg_word, names_filter=hook_name)[1][hook_name][0, -1]
    return a - b

steer = direction(" Love", " Hate")

def generate_steered(prompt, coef, n=3, max_new_tokens=25):
    def hook(resid, hook):
        resid[:, 1:] += coef * steer          # todas las posiciones salvo BOS
        return resid
    outs = []
    for s in range(n):
        torch.manual_seed(s)
        with model.hooks(fwd_hooks=[(hook_name, hook)]):
            outs.append(model.generate(prompt, max_new_tokens=max_new_tokens, temperature=0.7,
                                       use_past_kv_cache=False, verbose=False))
    return outs

prompt = "I went up to my friend and said"
for coef in [0, 1.5]:      # 🧪 prueba 0.5, 3, o un coeficiente negativo
    print(f"\n=== coef = {coef} ===")
    for o in generate_steered(prompt, coef): print("•", o.replace("\n", " "))

Con un coeficiente moderado el tono cambia; con uno demasiado alto el texto se degrada (empujamos el residual stream fuera de su distribución normal).

🧪 Prueba otros pares: `" wedding"`/`" funeral"`, `" Spain"`/`" France"`, `" happy"`/`" sad"`. Cambia `STEER_LAYER`: ¿en qué capas funciona mejor?

## 🧪 Retos

1. **Logit lens en castellano**: `logit_lens("La capital de Francia es", [" Paris", " Madrid", " la"])`. GPT-2 apenas vio castellano: ¿qué predice en cada capa? (Ojo: `logit_lens` solo acepta respuestas de **un** token; compruébalo con `model.to_str_tokens(" París")`.)
2. **Induction en texto real**: escribe un párrafo que repita un nombre inventado raro y comprueba con `plot_heads` si las induction heads que encontraste se activan.
3. **IOI con otros nombres**: cambia Mary/John por otros nombres de un solo token. ¿Se mantienen las mismas cabezas importantes?
4. **Patching por capas y componentes**: prueba `patching.get_act_patch_block_every(model, corrupt_tokens, clean_cache, metric)` y visualiza por separado resid / atención / MLP.

## Para seguir: superposición y SAEs

Las neuronas sueltas suelen ser **polisemánticas**: el modelo mete más conceptos que dimensiones tiene (*superposición*). Por eso hoy se usan *sparse autoencoders* para separar esas features.

In [ ]:
anim("superposition")

## Recursos

- **ARENA** (curso abierto de interpretabilidad): https://arena.education
- **Neuronpedia** (explorar features de SAEs en el navegador): https://neuronpedia.org
- Olsson et al. (2022), *In-context Learning and Induction Heads*.
- Wang et al. (2022), *Interpretability in the Wild: a Circuit for Indirect Object Identification in GPT-2 small*.
- Elhage et al. (2021), *A Mathematical Framework for Transformer Circuits*.
- Turner et al. (2023), *Steering Language Models With Activation Engineering*.